In [988]:
# IMPORTS FROM PYOMO
from pyomo.environ import (
    Block,
    ConcreteModel,
    Var,
    Param,
    Constraint,
    Objective,
    Expression,
    value,
    check_optimal_termination,
    assert_optimal_termination,
    TransformationFactory,
    units as pyunits,
)

from pyomo.util.check_units import assert_units_consistent
from pyomo.network import Arc


# IMPORTS FROM IDAES
from idaes.core import FlowsheetBlock, UnitModelCostingBlock
from idaes.models.unit_models import Feed, Product, StateJunction
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.util.scaling import calculate_scaling_factors, set_scaling_factor, constraint_scaling_transform
from idaes.core.util.initialization import propagate_state


# IMPORTS FROM WaterTAP
from watertap.property_models.NaCl_prop_pack import NaClParameterBlock
from watertap.property_models.seawater_prop_pack import SeawaterParameterBlock
from watertap.unit_models.pressure_changer import Pump
from watertap.unit_models.reverse_osmosis_0D import (
    ReverseOsmosis0D,
    ConcentrationPolarizationType,
    MassTransferCoefficient,
    PressureChangeType,
)
from watertap.unit_models.reverse_osmosis_1D import (
    ReverseOsmosis1D,
    PressureChangeType,
    MassTransferCoefficient,
    ConcentrationPolarizationType,
)

from watertap.core.solvers import get_solver

# TRANSLATOR FUNCTION
import idaes.logger as idaeslog
from idaes.core import declare_process_block_class
from idaes.core.util.exceptions import InitializationError
from idaes.models.unit_models.translator import TranslatorData

from watertap.core.util.model_diagnostics.infeasible import *
from idaes.core.util.model_diagnostics import DiagnosticsToolbox
import idaes.core.util.scaling as iscale

Low concentration inlet conditions and updated A value

In [989]:
mass_flow_water = 0.995 * pyunits.kg / pyunits.s

salt_concentration = 5 * pyunits.g / pyunits.L
density = 995 * pyunits.kg / pyunits.m**3
mass_flow_salt = pyunits.convert(salt_concentration * mass_flow_water / density, to_units=pyunits.kg / pyunits.s)

# Pump parameters
pump_efficiency = 0.85 * pyunits.dimensionless
operating_pressure = 10 * pyunits.bar

A_comp = 2.027e-11 * pyunits.m/(pyunits.s * pyunits.Pa)               # membrane water permeability coefficient [m/s-Pa]
B_comp = 3e-8 * pyunits.m/(pyunits.s)                              # membrane salt permeability coefficient [m/s]
membrane_area = 50  * pyunits.m**2
atmospheric = 101325  * pyunits.Pa
deltaP = -3 * pyunits.bar
channel_height = 1 * pyunits.mm 
spacer_porosity = 0.75  * pyunits.dimensionless
RR = 0.45   * pyunits.dimensionless

mass_flow_salt()

0.004999999999999998

In [990]:
m = ConcreteModel()
m.fs = FlowsheetBlock(dynamic=False)
m.fs.ro_properties = NaClParameterBlock()

m.fs.RO = ReverseOsmosis0D(
    property_package=m.fs.ro_properties,
    has_pressure_change=True,
    pressure_change_type=PressureChangeType.calculated,
    mass_transfer_coefficient=MassTransferCoefficient.calculated,
    concentration_polarization_type=ConcentrationPolarizationType.calculated,
    module_type="spiral_wound",
)

m.fs.RO.inlet.pressure.fix(operating_pressure)
m.fs.RO.inlet.temperature.fix(298)
m.fs.RO.inlet.flow_mass_phase_comp[0, "Liq","H2O"].fix(mass_flow_water)
m.fs.RO.inlet.flow_mass_phase_comp[0, "Liq","NaCl"].fix(mass_flow_salt)    

# Fix (2) membrane properties
m.fs.RO.A_comp.fix(A_comp)
m.fs.RO.B_comp.fix(B_comp)

# Fix (4) module specifications
m.fs.RO.feed_side.channel_height.fix(channel_height)
m.fs.RO.feed_side.spacer_porosity.fix(spacer_porosity)
m.fs.RO.area.fix(membrane_area)
m.fs.RO.deltaP.fix(deltaP)                           
# m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(RR)

# (1) outlet state variable
m.fs.RO.permeate.pressure[0].fix(atmospheric)

TransformationFactory("network.expand_arcs").apply_to(m)  

In [991]:
# Set scaling factors
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 10, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e4, index=("Liq", "NaCl"))

iscale.constraint_scaling_transform(m.fs.RO.eq_recovery_vol_phase[0.0], 1e2)
set_scaling_factor(m.fs.RO.area, 1e-3)

# Deal with low flows
m.fs.RO.feed_side.cp_modulus.setub(10)
m.fs.RO.feed_side.cp_modulus.setlb(1e-5)

m.fs.RO.flux_mass_phase_comp.setub(0.1)
m.fs.RO.flux_mass_phase_comp.setlb(1e-8)

m.fs.RO.feed_side.friction_factor_darcy.setub(200)
m.fs.RO.feed_side.K.setlb(1e-5)

calculate_scaling_factors(m)
print(f"dof = {degrees_of_freedom(m)}")

dof = 0


In [992]:
m.fs.RO.initialize()
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

print("Estimated area: ", value(m.fs.RO.area), pyunits.get_units(m.fs.RO.area))
print("Estimated recovery: ", value(m.fs.RO.recovery_vol_phase[0.0, 'Liq']), pyunits.get_units(m.fs.RO.recovery_vol_phase[0.0, 'Liq']))
print("Estimated inlet pressure: ", value(m.fs.RO.inlet.pressure[0]), pyunits.get_units(m.fs.RO.inlet.pressure[0]))

m.fs.RO.flux_mass_phase_comp.display()

2026-03-09 12:05:49 [INFO] idaes.init.fs.RO.feed_side: Initialization Complete
2026-03-09 12:05:49 [INFO] idaes.init.fs.RO: Initialization Complete: optimal - Optimal Solution Found
Estimated area:  50.0 m**2
Estimated recovery:  0.26329557569590517 dimensionless
Estimated inlet pressure:  1000000.0 Pa
flux_mass_phase_comp : Mass flux across membrane at inlet and outlet
    Size=4, Index=fs._time*fs.RO.feed_side.length_domain*fs.ro_properties.phase_list*fs.ro_properties.component_list, Units=kg/m**2/s
    Key                       : Lower : Value                  : Upper : Fixed : Stale : Domain
     (0.0, 0.0, 'Liq', 'H2O') : 1e-08 :   0.009127222415611543 :   0.1 : False : False :  Reals
    (0.0, 0.0, 'Liq', 'NaCl') : 1e-08 : 1.7157679289747888e-07 :   0.1 : False : False :  Reals
     (0.0, 1.0, 'Liq', 'H2O') : 1e-08 :  0.0013646516610720364 :   0.1 : False : False :  Reals
    (0.0, 1.0, 'Liq', 'NaCl') : 1e-08 : 2.0321138885680954e-07 :   0.1 : False : False :  Reals


Higher flowrate and updated membrane area

In [993]:
flow_vol = 500 * pyunits.m**3/pyunits.hour
density = 995 * pyunits.kg/pyunits.m**3

feed_conc = 5 * pyunits.g/pyunits.liter
membrane_area = 20000 * pyunits.m**2

mass_flow_water = pyunits.convert(flow_vol * density, to_units=pyunits.kg/pyunits.s) 
mass_flow_salt = pyunits.convert(feed_conc, to_units=pyunits.kg/pyunits.m**3) * pyunits.convert(flow_vol, to_units=pyunits.m**3/pyunits.s)

print("Mass flow of water: ", value(mass_flow_water), pyunits.get_units(mass_flow_water))
print("Mass flow of salt: ", value(mass_flow_salt), pyunits.get_units(mass_flow_salt))

Mass flow of water:  138.19444444444446 kg/s
Mass flow of salt:  0.6944444444444443 kg/s


In [994]:
m.fs.RO.inlet.flow_mass_phase_comp[0, "Liq","H2O"].fix(mass_flow_water)
m.fs.RO.inlet.flow_mass_phase_comp[0, "Liq","NaCl"].fix(mass_flow_salt)

m.fs.RO.area.fix(membrane_area)

m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e-2, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e1, index=("Liq", "NaCl"))
# set_scaling_factor(m.fs.RO.area, 1e-5)

calculate_scaling_factors(m)
degrees_of_freedom(m)

0

In [995]:
m.fs.RO.initialize()

2026-03-09 12:05:49 [INFO] idaes.init.fs.RO.feed_side: Initialization Complete
2026-03-09 12:05:49 [INFO] idaes.init.fs.RO: Initialization Complete: optimal - Optimal Solution Found


In [996]:
m.fs.RO.inlet.pressure.unfix()
m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(0.9)

In [997]:
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

print('degrees_of_freedom = ', degrees_of_freedom(m) )

print("\nEstimated area: ", value(m.fs.RO.area), pyunits.get_units(m.fs.RO.area))
print("Estimated recovery: ", value(m.fs.RO.recovery_vol_phase[0.0, 'Liq']), pyunits.get_units(m.fs.RO.recovery_vol_phase[0.0, 'Liq']))
print("Estimated inlet pressure: ", value(m.fs.RO.inlet.pressure[0]), pyunits.get_units(m.fs.RO.inlet.pressure[0]))
print("Estimated pressure drop: ", pyunits.convert(m.fs.RO.deltaP[0], to_units=pyunits.bar)() )
print('Membrane length: ', value(m.fs.RO.feed_side.length), pyunits.get_units(m.fs.RO.feed_side.length))

m.fs.RO.feed_side.dP_dx.display()
m.fs.RO.flux_mass_phase_comp.display()

degrees_of_freedom =  0

Estimated area:  20000.0 m**2
Estimated recovery:  0.9 dimensionless
Estimated inlet pressure:  1196754.5652946953 Pa
Estimated pressure drop:  -3.0000000000000004
Membrane length:  11.489237403444886 m
dP_dx : Pressure drop per unit length of channel at inlet and outlet
    Size=2, Index=fs._time*fs.RO.feed_side.length_domain, Units=kg/m**2/s**2
    Key        : Lower     : Value               : Upper   : Fixed : Stale : Domain
    (0.0, 0.0) : -200000.0 :  -48203.82472990599 : -1000.0 : False : False : NegativeReals
    (0.0, 1.0) : -200000.0 : -4018.9624691903214 : -1000.0 : False : False : NegativeReals
flux_mass_phase_comp : Mass flux across membrane at inlet and outlet
    Size=4, Index=fs._time*fs.RO.feed_side.length_domain*fs.ro_properties.phase_list*fs.ro_properties.component_list, Units=kg/m**2/s
    Key                       : Lower : Value                  : Upper : Fixed : Stale : Domain
     (0.0, 0.0, 'Liq', 'H2O') : 1e-08 :    0.0124455268608440